# Calculating Power-law scaling in fluctuations of information

In [1]:
import os
import numpy as np
import pandas as pd

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings;warnings.filterwarnings('ignore')

In [2]:
CORPUS_NAME = 'xling-'

In [3]:
DATA_PATH = '../data/ckpts/rosen'

OUTPUTS_PATH = 'reports'
if not os.path.exists(OUTPUTS_PATH):
    os.mkdir(OUTPUTS_PATH)

In [4]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [5]:
fs = [f for f in grab_all_files(DATA_PATH) if f.split('/')[-1].startswith(CORPUS_NAME)]
len(fs)

26

## Processing individual datasets

In [6]:
# formula = '{} ~ nx + ny + self'.format(COEF_VAR)

In [7]:
COEF_VAR = 'AVAR'

In [8]:
param_list = 'time_delta'

In [9]:
df_scale, df_regression, k_docs = [], [], dict()

In [10]:
for f in tqdm(fs):
    print('===][===')
    print(f.split('/')[-1])

    table = pq.ParquetFile(f)

    df = table.read(columns=[
        'corpus', 'conversation_id', 'x_speaker', 'y_speaker',
        COEF_VAR,
        'x_turn_id', 'y_turn_id',
        # 'nx', 'ny'
    ]).to_pandas()


    df['self'] = (df['x_speaker'] == df['y_speaker'])
    df['x_turn_id'] = [int(tid.split('-')[-1]) for tid in df['x_turn_id']]
    df[param_list] = (df['y_turn_id'] - df['x_turn_id']) + 1

    df = df.loc[
        (~df[COEF_VAR].isna())
        & (~df[param_list].isna())
    ]

    for corpus in tqdm(df['corpus'].unique()):

        if corpus not in k_docs.keys():
            k_docs[corpus] = {
                True: 0,
                False: 0
            }

        sub = df.loc[df['corpus'].isin([corpus])]

        groups = sub.groupby(by=['conversation_id', 'x_speaker', 'self'])
        for labels, dfi in tqdm(groups):

            if len(dfi):
                k_docs[corpus][labels[2]] += 1

                b,a = np.polyfit(np.log(dfi[param_list].values), np.log(dfi['AVAR'].values + 1e-9), 1)

                df_regression += [
                    {
                        'corpus': corpus,
                        'cid': labels[0],
                        'speaker': labels[1],
                        'self': labels[2],
                        'a': np.exp(a),
                        'b': b,
                        'k': len(dfi)
                    }
                ]

df_regression = pd.DataFrame(df_regression)

  0%|          | 0/26 [00:00<?, ?it/s]

===][===
xling-callfriend-49.parquet


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/240 [00:00<?, ?it/s]

===][===
xling-callhome-149.parquet


...

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/202 [00:00<?, ?it/s]

===][===
xling-callhome-399.parquet


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/208 [00:00<?, ?it/s]

In [11]:
df_regression['lang'] = [c.split('-')[1] if (c != 'yiddish') else 'yid' for c in df_regression['corpus']]

In [12]:
# df_regression.isna().sum()

In [13]:
# df_regression = df_regression.loc[
#     (df_regression.isna().astype(int).sum(axis=1) == 0)
#     & (np.isinf(df_regression).astype(int).sum(axis=1) == 0)
# ]

In [14]:
if not os.path.exists(os.path.join(OUTPUTS_PATH,'all.csv')):
    df_regression.to_csv(
        os.path.join(OUTPUTS_PATH,'all.csv'),
        index=False,
        encoding='utf-8'
    )

else:
    df_regression.to_csv(
        os.path.join(OUTPUTS_PATH,'all.csv'),
        index=False,
        header=False,
        encoding='utf-8',
        mode='a'
    )

## Corpus level analysis of results

In [15]:
split_by = ['lang', 'corpus', 'self']

In [16]:
df0 = df_regression[split_by+['b']].groupby(by=split_by).agg('mean').reset_index()
df0['std'] = df_regression[split_by+['b']].groupby(by=split_by).agg('std').reset_index()['b'].values

df0['k'] = [k_docs[corpus][s] for corpus, s in df0[['corpus', 'self']].values]

df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [17]:
df0['t'] = df0['b'] / df0['se']

In [18]:
df0.head(n=50)

,lang,corpus,self,b,std,k,se,t
0,cro,croation-cro,False,-0.345926,0.267521,595,0.010967,-31.541585
1,cro,croation-cro,True,-0.415675,0.529010,574,0.022080,-18.825511
2,deu,callhome-deu,False,-0.418759,0.186926,260,0.011593,-36.122844
3,deu,callhome-deu,True,-0.436699,0.393506,258,0.024499,-17.825466
4,eng,callfriend-eng-n,False,-0.316660,0.266850,69,0.032125,-9.857136
5,eng,callfriend-eng-n,True,-0.472274,0.404599,68,0.049065,-9.625510
6,eng,callfriend-eng-s,False,-0.305004,0.372074,25,0.074415,-4.098698
7,eng,callfriend-eng-s,True,-0.383329,0.271774,23,0.056669,-6.764364
8,eng,callhome-eng,False,-0.388320,0.197603,369,0.010287,-37.749388
9,eng,callhome-eng,True,-0.431194,0.305384,358,0.016140,-26.715780


In [19]:
df0.to_csv(
    os.path.join(OUTPUTS_PATH, CORPUS_NAME+'.csv'),
    index=False, encoding='utf-8'
)